# Rotten Tomatoes review scraper

I used this notebook during my thesis to collect audience reviews from Rotten Tomatoes. The scraper loads the review cards for each movie and extracts the review text, date, star rating and whether the review is marked as verified.

The movie list was stored separately and included the IMDb ID and Rotten Tomatoes URL for each title. This is a cleaned copy of the thesis code. Rotten Tomatoes can change its page structure, so I am keeping it here to document the data-collection process rather than claiming that it will still run unchanged today.


## Setup

In [ ]:
import csv
import os
import re
from typing import Any, Dict, List, Optional

import pandas as pd
from playwright.async_api import Page, async_playwright

CSV_HEADERS = [
    "movie_title",
    "movie_release_date",
    "genres",
    "review_date",
    "review_text",
    "star_rating",
    "is_verified",
    "author",
]

MONTHS = {
    month: number
    for number, month in enumerate(
        ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
         "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"],
        start=1,
    )
}


def clean_text(value: Optional[str]) -> str:
    return re.sub(r"\s+", " ", value or "").strip()


def to_iso_date(value: Optional[str]) -> str:
    value = (value or "").strip()
    match = re.search(
        r"(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+(\d{1,2}),\s*(\d{4})",
        value,
        re.I,
    )

    if not match:
        return value

    month = MONTHS[match.group(1)[:3].title()]
    day = int(match.group(2))
    year = int(match.group(3))
    return f"{year:04d}-{month:02d}-{day:02d}"


## Page handling

In [ ]:
CONSENT_HOST_HINTS = [
    "onetrust",
    "cookielaw",
    "fundingchoices",
    "privacy-mgmt",
    "trustarc",
    "quantcast",
]

CONSENT_SELECTORS = [
    "button:has-text('Accept')",
    "button:has-text('I Agree')",
    "button:has-text('Agree')",
    "button:has-text('Allow all')",
    "button:has-text('Yes, I agree')",
    "button[aria-label*='accept' i]",
    "[aria-label*='accept' i]",
    "[aria-label*='agree' i]",
]


async def click_if_visible(scope, selector) -> bool:
    try:
        element = await scope.query_selector(selector)
        if element and await element.is_visible():
            await element.click(force=True, delay=20)
            return True
    except Exception:
        pass

    return False


async def handle_consent_and_overlays(page: Page):
    # Consent banners sometimes blocked the review controls during collection.
    try:
        await page.unroute("**/*")
    except Exception:
        pass

    await page.route(
        "**/*",
        lambda route: (
            route.abort()
            if any(host in route.request.url.lower() for host in CONSENT_HOST_HINTS)
            else route.continue_()
        ),
    )

    for selector in CONSENT_SELECTORS:
        if await click_if_visible(page, selector):
            await page.wait_for_timeout(150)

    try:
        for frame in page.frames:
            frame_url = (frame.url or "").lower()
            if not any(host in frame_url for host in CONSENT_HOST_HINTS):
                continue

            clicked = False
            for selector in CONSENT_SELECTORS:
                if await click_if_visible(frame, selector):
                    clicked = True
                    await page.wait_for_timeout(150)
                    break

            if not clicked:
                try:
                    await page.evaluate(
                        """
                        (() => {
                          document.querySelectorAll('iframe').forEach(frame => {
                            const src = (frame.getAttribute('src') || '').toLowerCase();
                            if (/onetrust|cookielaw|fundingchoices|privacy-mgmt|trustarc|quantcast/.test(src)) {
                              frame.remove();
                            }
                          });
                        })()
                        """
                    )
                except Exception:
                    pass
    except Exception:
        pass

    # Remove any full-screen elements that still cover the page.
    try:
        await page.evaluate(
            """
            (() => {
              const hide = element => {
                element.style.setProperty('display', 'none', 'important');
                element.remove?.();
              };

              const isOverlay = element => {
                const style = getComputedStyle(element);
                const zIndex = parseInt(style.zIndex || '0', 10);
                return (style.position === 'fixed' || style.position === 'sticky') && zIndex >= 1000;
              };

              document.querySelectorAll('div, section, aside, header, [role="dialog"]').forEach(element => {
                if (isOverlay(element) || /overlay|interstitial|backdrop|modal/i.test(element.className)) {
                  hide(element);
                }
              });
            })()
            """
        )
    except Exception:
        pass


async def load_more_until_plateau(
    page: Page,
    plateau_rounds: int = 3,
    max_iters: int = 10,
) -> int:
    previous_count = await page.locator("review-card").count()
    rounds_without_growth = 0

    for _ in range(max_iters):
        try:
            await page.mouse.wheel(0, 2500)
        except Exception:
            pass

        await page.wait_for_timeout(800)
        load_more = page.get_by_text("LOAD MORE", exact=False)

        try:
            if await load_more.count() > 0 and await load_more.first.is_visible():
                await load_more.first.click(force=True, timeout=4000)
                await page.wait_for_timeout(1500)
        except Exception:
            pass

        current_count = await page.locator("review-card").count()

        if current_count > previous_count:
            previous_count = current_count
            rounds_without_growth = 0
        else:
            rounds_without_growth += 1

        if rounds_without_growth >= plateau_rounds:
            break

    return previous_count


SHOW_MORE_BUTTON = "button:has-text('Show More'), button:has-text('SHOW MORE')"


async def expand_truncated_reviews(page: Page, max_passes: int = 3):
    for _ in range(max_passes):
        clicked_any = False
        buttons = page.locator(SHOW_MORE_BUTTON)

        for i in range(await buttons.count()):
            try:
                button = buttons.nth(i)
                if await button.is_visible():
                    await button.click(timeout=800)
                    clicked_any = True
                    await page.wait_for_timeout(150)
            except Exception:
                pass

        if not clicked_any:
            break


## Extract the review fields

In [ ]:
async def extract_movie_meta(page: Page) -> Dict[str, str]:
    script = r"""
    (() => {
      const clean = value => (value || "").replace(/\s+/g, " ").trim();
      const result = { title: "", date: "", genres: [] };

      const titleSelectors = [
        "a.sidebar-title",
        "[data-qa='score-panel-movie-title']",
        "[data-qa='score-panel-media-title']",
        "h1[data-qa*='title']",
        "h1"
      ];

      for (const selector of titleSelectors) {
        const node = document.querySelector(selector);
        if (node && clean(node.textContent)) {
          result.title = clean(node.textContent).replace(/Reviews$/i, "").trim();
          break;
        }
      }

      if (!result.title && document.title) {
        result.title = clean(document.title)
          .replace(/Reviews.*$/i, "")
          .replace(/- Rotten Tomatoes.*$/i, "")
          .trim();
      }

      const dateSelectors = [
        "[data-qa*='movie-info'] time",
        "time[datetime]",
        "p time",
        "li time"
      ];

      for (const selector of dateSelectors) {
        const node = document.querySelector(selector);
        if (node && clean(node.textContent)) {
          result.date = clean(node.textContent);
          break;
        }
      }

      const genreSelectors = [
        "[data-qa*='genre']",
        "[data-qa='genres'] a",
        "a[href*='/browse/movies_'][href*='/genre/']",
        ".genre, .genres a"
      ];

      for (const selector of genreSelectors) {
        const values = Array.from(document.querySelectorAll(selector))
          .map(node => clean(node.textContent))
          .filter(Boolean);

        if (values.length) {
          result.genres = values;
          break;
        }
      }

      return result;
    })()
    """

    data = await page.evaluate(script)
    movie_title = clean_text(data.get("title"))
    release_date = clean_text(data.get("date") or "")
    genres = "|".join(
        {
            clean_text(genre)
            for genre in (data.get("genres") or [])
            if clean_text(genre)
        }
    )

    return {
        "movie_title": movie_title,
        "movie_release_date": to_iso_date(release_date) or release_date,
        "genres": genres,
    }


async def extract_reviews(page: Page) -> List[Dict[str, Any]]:
    script = r"""
    (() => {
      const clean = value => (value || "").replace(/\s+/g, " ").trim();
      const results = [];
      const cards = Array.from(document.querySelectorAll("review-card"));

      for (const card of cards) {
        const fullText = clean(card.textContent || "");

        let reviewText = "";
        const textElement =
          card.querySelector("drawer-more[slot='review'] [slot='content']") ||
          card.querySelector("drawer-more[slot='review']") ||
          card.querySelector("[data-qa='review-text']") ||
          card;

        if (textElement) {
          reviewText = clean(textElement.textContent || "");
        }
        if (!reviewText) {
          continue;
        }

        let header = "";
        const reviewStart = fullText.indexOf(reviewText);
        if (reviewStart > 0) {
          header = clean(fullText.slice(0, reviewStart));
        }

        let author = "";
        if (header) {
          const authorMatch = header.match(
            /^(.+?)\s+((Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)\b|\d{1,2}\/\d{1,2}\/\d{2,4})/i
          );
          if (authorMatch) {
            author = clean(authorMatch[1]);
          }
        }

        let date = "";
        const dateElement =
          card.querySelector("span[slot='timestamp']") ||
          card.querySelector("[data-qa='review-date']") ||
          card.querySelector("time") ||
          card.querySelector("span");

        if (dateElement) {
          date = clean(dateElement.textContent || "");
        }

        if (!date && header) {
          const dateMatch = header.match(
            /((Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)\s+\d{1,2},?\s+\d{2,4}|\d{1,2}\/\d{1,2}\/\d{2,4})/i
          );
          if (dateMatch) {
            date = clean(dateMatch[1]);
          }
        }

        let verified = false;

        if (header && /verified/i.test(header)) {
          verified = true;
        }

        const verifiedBadge = card.querySelector(".verified-review-badge");
        if (!verified && verifiedBadge && /verified/i.test(clean(verifiedBadge.textContent))) {
          verified = true;
        }

        const verifiedAttribute = card.getAttribute("verified-review");
        if (!verified && verifiedAttribute && verifiedAttribute.toLowerCase() === "true") {
          verified = true;
        }

        let stars = null;

        const ratingStars =
          card.querySelector("rating-stars-group[slot='rating']") ||
          card.querySelector("rating-stars-group");

        if (ratingStars && ratingStars.getAttribute("score") !== null) {
          const score = parseFloat(ratingStars.getAttribute("score"));
          if (!Number.isNaN(score)) {
            stars = score;
          }
        }

        if (stars === null) {
          const ratingNode =
            card.querySelector("[aria-label*='out of']") ||
            card.querySelector("score-icon-variant");

          if (ratingNode) {
            const label =
              ratingNode.getAttribute("aria-label") ||
              clean(ratingNode.textContent || "");
            const ratingMatch = label.match(/(\d+(?:\.\d+)?)\s*out of\s*5/i);

            if (ratingMatch) {
              const score = parseFloat(ratingMatch[1]);
              if (!Number.isNaN(score)) {
                stars = score;
              }
            }
          }
        }

        results.push({
          author,
          date,
          reviewText,
          stars,
          verified
        });
      }

      return results;
    })()
    """

    data = await page.evaluate(script)

    rows = []
    for item in data:
        rows.append(
            {
                "author": clean_text(item.get("author") or ""),
                "review_date": (
                    to_iso_date(item.get("date") or "")
                    or clean_text(item.get("date") or "")
                ),
                "review_text": clean_text(item.get("reviewText") or ""),
                "star_rating": (
                    item.get("stars")
                    if item.get("stars") not in (None, "")
                    else ""
                ),
                "is_verified": 1 if item.get("verified") else 0,
            }
        )

    return rows


## Scrape one movie

In [ ]:
async def scrape_rt_reviews(
    url: str,
    out_csv: Optional[str] = None,
    max_clicks: int = 10,
    keep_open: bool = False,
) -> Dict[str, Any]:

    if not isinstance(url, str) or not url.lower().startswith(("http://", "https://")):
        raise ValueError(f"Invalid URL: {url!r}")

    playwright = await async_playwright().start()
    browser = await playwright.chromium.launch(headless=False)

    context = await browser.new_context(
        viewport={"width": 1366, "height": 900},
        user_agent=(
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124 Safari/537.36"
        ),
        locale="en-US",
        timezone_id="America/Los_Angeles",
        extra_http_headers={"Accept-Language": "en-US,en;q=0.9"},
    )

    page = await context.new_page()

    try:
        await page.goto(url, wait_until="domcontentloaded", timeout=60000)
        await handle_consent_and_overlays(page)

        # The review cards sometimes only appeared after the page had been scrolled.
        for _ in range(3):
            try:
                await page.evaluate("window.dispatchEvent(new Event('scroll'))")
                await page.mouse.wheel(0, 2500)
                await page.set_viewport_size({"width": 1366, "height": 901})
                await page.set_viewport_size({"width": 1366, "height": 900})
            except Exception:
                pass

            await page.wait_for_timeout(800)

        first_count = 0
        for _ in range(12):
            first_count = await page.locator("review-card").count()
            if first_count > 0:
                break

            await page.mouse.wheel(0, 2000)
            await page.wait_for_timeout(900)

        if first_count == 0:
            message = "No Rotten Tomatoes review cards were found."

            if keep_open:
                return {
                    "playwright": playwright,
                    "browser": browser,
                    "context": context,
                    "page": page,
                    "rows": [],
                }

            await context.close()
            await browser.close()
            await playwright.stop()
            raise TimeoutError(message)

        total_reviews = await load_more_until_plateau(
            page,
            plateau_rounds=3,
            max_iters=max_clicks,
        )
        print(f"Loaded around {total_reviews} review cards.")

        await expand_truncated_reviews(page)

        movie_meta = await extract_movie_meta(page)
        reviews = await extract_reviews(page)

        rows = [
            {
                "movie_title": movie_meta["movie_title"],
                "movie_release_date": movie_meta["movie_release_date"],
                "genres": movie_meta["genres"],
                "review_date": review["review_date"],
                "review_text": review["review_text"],
                "star_rating": review["star_rating"],
                "is_verified": review["is_verified"],
                "author": review["author"],
            }
            for review in reviews
        ]

        if out_csv:
            os.makedirs(os.path.dirname(out_csv) or ".", exist_ok=True)
            with open(out_csv, "w", newline="", encoding="utf-8") as file:
                writer = csv.DictWriter(file, fieldnames=CSV_HEADERS)
                writer.writeheader()
                writer.writerows(rows)

        if keep_open:
            return {
                "playwright": playwright,
                "browser": browser,
                "context": context,
                "page": page,
                "rows": rows,
            }

        await context.close()
        await browser.close()
        await playwright.stop()
        return {"rows": rows}

    except Exception:
        if not keep_open:
            try:
                await context.close()
            except Exception:
                pass
            try:
                await browser.close()
            except Exception:
                pass
            try:
                await playwright.stop()
            except Exception:
                pass

        raise


## Run the scraper for the movie list

In [ ]:
# The lookup file stays outside the public repository.
INPUT_CSV = "../data/private/movie_lookup.csv"
movies_df = pd.read_csv(INPUT_CSV)

print(f"Loaded {len(movies_df)} movies.")
display(movies_df[["title", "year", "rt_url"]].head())

all_rt_rows = []

for index, movie in movies_df.iterrows():
    rt_url = str(movie["rt_url"]).strip()
    imdb_id = str(movie.get("imdb_id", "")).strip()
    movie_title = movie.get("title", "")
    movie_year = movie.get("year", "")

    print(f"\n{index + 1}/{len(movies_df)}: {movie_title} ({movie_year})")

    if not rt_url:
        print("Skipping because the Rotten Tomatoes URL is missing.")
        continue

    try:
        result = await scrape_rt_reviews(
            url=rt_url,
            out_csv=None,
            max_clicks=15,
            keep_open=False,
        )

        rows = result["rows"]

        # Keep the IMDb ID from the lookup file so the sources can be joined later.
        for row in rows:
            row["imdb_id"] = imdb_id
            row["csv_year"] = movie_year

        all_rt_rows.extend(rows)
        print(f"Collected {len(rows)} reviews.")

    except Exception as error:
        print(f"Could not scrape this movie: {error}")

print(f"\nFinished with {len(all_rt_rows):,} reviews in total.")


## Save the result

In [ ]:
rt_reviews_df = pd.DataFrame(all_rt_rows)

column_order = [
    "imdb_id",
    "movie_title",
    "csv_year",
    "movie_release_date",
    "genres",
    "review_date",
    "review_text",
    "star_rating",
    "is_verified",
    "author",
]

column_order = [
    column
    for column in column_order
    if column in rt_reviews_df.columns
]
rt_reviews_df = rt_reviews_df[column_order]

# Raw scraped reviews stay outside the public repository.
OUTPUT_CSV = "../data/private/rt_reviews.csv"
rt_reviews_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print(f"Saved {len(rt_reviews_df):,} reviews to {OUTPUT_CSV}.")
rt_reviews_df.head()
